<a href="https://colab.research.google.com/github/QidiXu96/Interactive-AI-for-Identifying-Topics/blob/main/topic_identification_covid19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import json
import time
import os
from docx import Document
from openai import AzureOpenAI
import re

In [4]:
client = AzureOpenAI(
         azure_endpoint = "your_azure_endpoint",
         api_key = "your_api_key",
         api_version = "your_api_version"
         )

In [5]:
def get_completion(messages):
    """ GET completion from openai api"""
    response = client.chat.completions.create(
        model = "your_model",
        messages = messages,
        max_tokens = 6000,
        temperature = 0.7,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0
)
    return response

In [6]:
def calculate_cost_from_response(response, price_per_1000_input_tokens = 0.00015, price_per_1000_output_tokens = 0.0006):
    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens

    input_cost = (input_tokens / 1000) * price_per_1000_input_tokens
    output_cost = (output_tokens / 1000) * price_per_1000_output_tokens

    total_cost = input_cost + output_cost

    return total_cost

In [7]:
def extract_text_from_docx(file_path):
    doc = Document(file_path)
    full_text = []
    for paragraph in doc.paragraphs:
        full_text.append(paragraph.text)
    return '\n'.join(full_text)

### Training Process -- Finetune clue and reasoning prompt

In [8]:
def clue_prompt(dialogue, topics, clue_instruction):
    prompt = (
        f"{clue_instruction}\n\n"
        f"Dialogue:\n{dialogue}\n\n"
        f"Topics:\n{topics}\n\n"
        f"Clues: "
    )

    messages = [
        {
            "role": "system",
            "content": """
            You are a qualitative research expert with extensive experience analyzing interviews.
            These interviews were conducted with health workers, policymakers, key informants, and patients between Oct-Dec 2020 to examining the health system response to COVID-19 in Sierra Leone.
            The research aims to explore how the pandemic affected service delivery, health workers, patient access to services, leadership, and governance. Additionally, the research examines to what extent the legacy of the 2013–2016 Ebola outbreak influenced the COVID-19 response and public perception.
            Your task is to extract key clues (limit to 200 words) diectly from original dialogues supporting each given identified topic.

            Clues must:
            - Be direct quotes from the dialogue (no summarization or interpretation).
            - Be brief but contextually complete.
            - Be relevant to the identified topics.
            """
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
    return messages

In [9]:
def reasoning_prompt(clues, topics, reasoning_instruction):
    prompt = (
        f"{reasoning_instruction}\n\n"
        f"Clues:\n{clues}\n\n"
        f"Topics:\n{topics}\n\n"
        f"Reasonings: "
        )

    messages = [
        {
            "role": "system",
            "content": """
            You are a qualitative research expert with extensive experience analyzing interviews.
            These interviews were conducted with health workers, policymakers, key informants, and patients between Oct-Dec 2020 to examining the health system response to COVID-19 in Sierra Leone.
            The research aims to explore how the pandemic affected service delivery, health workers, patient access to services, leadership, and governance. Additionally, the research examines to what extent the legacy of the 2013–2016 Ebola outbreak influenced the COVID-19 response and public perception.
            Your goal is to provide a clear and concise reasoning process (limit to 150 words) based on provided clues to explain each corresponding identified topic.

            Ensure your reasoning:
            - Links the clues directly to the topic.
            - Explains the logical connection between the clues and topic.
            - Avoids adding external context or information not present in the clues.
            """
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    return messages

In [10]:
def evaluation_prompt_batch(json_file_paths):
    prompt = ""

    for i, file_path in enumerate(json_file_paths):
        with open(file_path, "r") as file:
            data = json.load(file)
            clues = data.get("clues", "N/A")
            reasoning = data.get("reasoning", "N/A")
            topics = data.get("topics", "N/A")

        # Append each dialogue's evaluation content to the prompt
        prompt += (
            f"### Clues-Reasoning-Topic Pair {i + 1} ###\n\n"
            f"**Clues:** {clues}\n\n"
            f"**Reasoning:** {reasoning}\n\n"
            f"**Topics:** {topics}\n\n"
            "### Evaluation Task ###\n"
            "For the above Clues-Reasoning-Topic pair:\n"
            "1. **Clue Quality:** Evaluate the clues based on the following:\n"
            "   - How relevant and accurate are the clues in supporting the topic(s)?\n"
            "   - Are the clues complete (include all key information) and free of irrelevant details?\n"
            "   - Do the clues contain context or are they missing critical information from the dialogue?\n\n"
            "2. **Reasoning Quality:** Assess the reasoning based on the following:\n"
            "   - Does the reasoning logically connect the clues to the topic(s)?\n"
            "   - Are there any gaps or missing logic in the reasoning process?\n"
            "   - Is the reasoning concise and free of unnecessary content?\n\n"
        )

    # Add the aggregate feedback section
    prompt += (
        "### Aggregate Feedback Task ###\n"
        "Based on your evaluation of all the Clues-Reasoning-Topic pairs, provide:\n\n"
        "**Common Issues:**\n"
        "- **Clue Generation:** Identify recurring problems in the generated clues (e.g., missing context, irrelevant clues).\n"
        "- **Reasoning Generation:** Highlight frequent issues in reasoning (e.g., logical gaps, weak connections between clues and topics).\n\n"
        "**Suggestions for Improvement:**\n"
        "- **Clue Prompt:** Propose specific improvements to the clue generation prompt.\n"
        "- **Reasoning Prompt:** Recommend actionable enhancements to the reasoning generation prompt.\n\n"
    )

    messages = [
        {
            "role": "system",
            "content": """
            You are an evaluation expert tasked with analyzing a BATCH of clue-reasoning-topic pairs.
            These interviews were conducted with health workers, policymakers, key informants, and patients between Oct-Dec 2020 to examining the health system response to COVID-19 in Sierra Leone.
            The research aims to explore how the pandemic affected service delivery, health workers, patient access to services, leadership, and governance. Additionally, the research examines to what extent the legacy of the 2013–2016 Ebola outbreak influenced the COVID-19 response and public perception.

            Your tasks are as follows:\n
            1. Evaluate each Clues-Reasoning-Topic pair for Clue Quality and Reasoning Quality.\n
            2. Provide feedback on both the relevance and completeness of the clues, and the logical coherence of the reasoning.\n
            3. Identify common issues across all pairs in clue and reasoning generation.\n
            4. Suggest improvements to the clue and reasoning prompts based on recurring patterns of errors.
            """
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    return messages

In [11]:
def optimization_prompt(clue_prompt, reasoning_prompt, feedback):
    prompt = (
        "You are tasked with improving two prompts based on provided feedback.\n\n"
        "### Task Description ###\n"
        "Given the following feedback, improve both the **Clue Prompt** and the **Reasoning Prompt** simultaneously to address the issues and suggestions provided:\n\n"
        "1. The **Clue Prompt** should:\n"
        "   - Guide the user/system to extract relevant, precise, and contextually complete clues directly from the dialogue.\n"
        "   - Ensure the clues are accurate quotes, avoid irrelevant or incomplete clues, and incorporate missing elements identified in the feedback.\n"
        "   - Focus on ensuring clarity and usability of the prompt.\n\n"
        "2. The **Reasoning Prompt** should:\n"
        "   - Guide the user/system to logically and effectively connect the clues to the identified topics.\n"
        "   - Ensure the reasoning structure is clear, addresses logical gaps, and builds a strong link between the clues and topics.\n"
        "   - Incorporate improvements to reasoning clarity and structure as per the feedback.\n\n"
        "### Provided Inputs ###\n"
        f"**Feedback:**\n{feedback}\n\n"
        f"**Current Clue Prompt:**\n{clue_prompt}\n\n"
        f"**Current Reasoning Prompt:**\n{reasoning_prompt}\n\n"
        "### Output Instructions ###\n"
        "You MUST provide your improved prompts formatted as follows:\n"
        "- For the clue prompt: `<IMPROVED_CLUE_PROMPT> your improved clue prompt text </IMPROVED_CLUE_PROMPT>`\n"
        "- For the reasoning prompt: `<IMPROVED_REASONING_PROMPT> your improved reasoning prompt text </IMPROVED_REASONING_PROMPT>`\n\n"
        "The text provided between these tags will directly replace the current prompts, so ensure your improvements are complete, clear, and directly address the feedback provided.\n\n"
    )

    messages = [
        {
            "role": "system",
            "content": """
            You are part of an optimization system that improves text. You will be asked to creatively and critically improve the clue prompt and reasoing prompt (instructions).
            You will receive some feedback, and use the feedback to improve both clue and reasoning prompts simultaneously. The feedback may be noisy, identify what is important and what is correct.
            Pay attention to the role description of the clue and reasoning prompts (instructions), and the context in which it is used.
            """

        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    return messages

In [12]:
def complete_workflow(docx_file_paths, topics_list, clue_instruction, reasoning_instruction, json_output_dir, iteration, price_per_1000_input_tokens=0.00015, price_per_1000_output_tokens=0.0006):
    json_file_paths = []
    total_cost = 0.0

    for i, file_path in enumerate(docx_file_paths):
        try:
            # Step 1: Extract dialogue from `.docx`
            dialogue = extract_text_from_docx(file_path)

            # Get the corresponding topics
            topics = topics_list[i]

            # Step 2: Generate clues using `clue_prompt`
            clue_messages = clue_prompt(dialogue, topics, clue_instruction)
            clue_response = get_completion(clue_messages)
            clue_cost = calculate_cost_from_response(clue_response, price_per_1000_input_tokens, price_per_1000_output_tokens)
            total_cost += clue_cost
            clues = clue_response.choices[0].message.content.strip()

            # Step 3: Generate reasoning using `reasoning_prompt`
            reasoning_messages = reasoning_prompt(clues, topics, reasoning_instruction)
            reasoning_response = get_completion(reasoning_messages)
            reasoning_cost = calculate_cost_from_response(reasoning_response, price_per_1000_input_tokens, price_per_1000_output_tokens)
            total_cost += reasoning_cost
            reasoning = reasoning_response.choices[0].message.content.strip()

            # Save intermediate results to JSON
            json_file_path = f"{json_output_dir}/{file_path.split('/')[-1].replace('.docx', f'_iteration_{iteration + 1}.json')}"
            json_file_paths.append(json_file_path)
            with open(json_file_path, "w") as json_file:
                json.dump({
                    "dialogue": file_path,
                    "clues": clues,
                    "reasoning": reasoning,
                    "topics": topics
                }, json_file, indent=4)

        except Exception as e:
            print(f"Error processing {file_path}: {e}")

    # Step 4: Evaluate batches using `evaluation_prompt_batch`
    evaluation_messages = evaluation_prompt_batch(json_file_paths)
    evaluation_response = get_completion(evaluation_messages)
    evaluation_cost = calculate_cost_from_response(evaluation_response, price_per_1000_input_tokens, price_per_1000_output_tokens)
    total_cost += evaluation_cost

    aggregate_feedback_marker = "Aggregate Feedback"
    agg_feedback = ""
    if aggregate_feedback_marker in evaluation_response.choices[0].message.content:
        agg_feedback = evaluation_response.choices[0].message.content.split(aggregate_feedback_marker, 1)[1]

    # Step 5: Optimize prompts using `optimization_prompt`
    optimized_messages = optimization_prompt(clue_instruction, reasoning_instruction, agg_feedback)
    optimization_response = get_completion(optimized_messages)
    optimization_cost = calculate_cost_from_response(optimization_response, price_per_1000_input_tokens, price_per_1000_output_tokens)
    total_cost += optimization_cost
    optimized_clue_prompt = optimization_response.choices[0].message.content.split("<IMPROVED_CLUE_PROMPT>")[1].split("</IMPROVED_CLUE_PROMPT>")[0].strip()
    optimized_reasoning_prompt = optimization_response.choices[0].message.content.split("<IMPROVED_REASONING_PROMPT>")[1].split("</IMPROVED_REASONING_PROMPT>")[0].strip()

    # Output the results for this iteration
    return {
        "feedback": agg_feedback,
        "optimized_clue_prompt": optimized_clue_prompt,
        "optimized_reasoning_prompt": optimized_reasoning_prompt,
        "total_cost": total_cost
    }

In [13]:
def iterative_workflow(
    docx_file_paths, topics_list, initial_clue_instruction, initial_reasoning_instruction, json_output_dir, num_iterations, price_per_1000_input_tokens=0.00015, price_per_1000_output_tokens=0.0006
):
    clue_instruction = initial_clue_instruction
    reasoning_instruction = initial_reasoning_instruction
    final_results = {}
    total_cost = 0.0

    for iteration in range(num_iterations):
        print(f"Running iteration {iteration + 1}/{num_iterations}...")
        iteration_json_output_dir = f"{json_output_dir}/iteration_{iteration + 1}"
        os.makedirs(iteration_json_output_dir, exist_ok=True)

        # Step 1: Run the complete workflow for this iteration
        results = complete_workflow(docx_file_paths, topics_list, clue_instruction, reasoning_instruction, iteration_json_output_dir, iteration, price_per_1000_input_tokens, price_per_1000_output_tokens)

        # Update the prompts for the next iteration
        clue_instruction = results["optimized_clue_prompt"]
        reasoning_instruction = results["optimized_reasoning_prompt"]

        total_cost += results.get("total_cost", 0.0)

        final_results[f"Iteration {iteration + 1}"] = {
            "clue_prompt": clue_instruction,
            "reasoning_prompt": reasoning_instruction,
            "feedback": results["feedback"],
            "iteration_cost": results.get("total_cost", 0.0)
        }

        print(f"\nIteration {iteration + 1} Feedback:")
        print(results["feedback"])
        print(f"\nOptimized Clue Prompt (Iteration {iteration + 1}):")
        print(clue_instruction)
        print(f"\nOptimized Reasoning Prompt (Iteration {iteration + 1}):")
        print(reasoning_instruction)
        print(f"\nIteration {iteration + 1} Cost: ${results.get('total_cost', 0.0):.4f}")

    return {
        "final_results": final_results,
        "total_cost": total_cost
    }

To save computational cost, here we only use two interviews to finetune prompts

In [16]:
# this one as the final training version
docx_file_paths = [
    "/content/interview/Transcript_A.docx",
    "/content/interview/Transcript_E.docx"
]

topics_list = [
    "1. No drugs, 2. Patient experience about no money to travel, 3. Patient experience about feeling isolated",
    "1. Learning from Ebola so Covid-19 isn't a new thing, 2. Fear of contracting COVID-19"
]

initial_clue_instruction = "List clues (i.e. key phrases, contextual information, semantic and emotional tones, temporal information) in the following interviews that support each given identified topic.\n\n"

initial_reasoning_instruction = "Based on the given clues, generate the reasoning process that supports the identified topics.\n\n"

json_output_dir = "/content/training_results"

num_iterations = 3

final_results = iterative_workflow(
        docx_file_paths, topics_list, initial_clue_instruction, initial_reasoning_instruction, json_output_dir, num_iterations, price_per_1000_input_tokens=0.00015, price_per_1000_output_tokens=0.0006
    )

# Print final optimized prompts and feedback
final_iteration_results = final_results["final_results"][f"Iteration {num_iterations}"]

Running iteration 1/3...

Iteration 1 Feedback:


**Common Issues:**
- **Clue Generation:**
  - Clues often lack clarity and specificity; this can lead to confusion about the exact issues being highlighted.
  - Some clues contain irrelevant details or are poorly structured, making it difficult to extract the main point.
  
- **Reasoning Generation:**
  - Reasoning sometimes repeats ideas unnecessarily or lacks depth in connecting clues to broader systemic issues.
  - There are gaps in the rationale that could benefit from additional contextual information or examples.

**Suggestions for Improvement:**
- **Clue Prompt:**
  - Encourage the generation of more specific and structured clues that directly address the topic. For example, ask for clear examples of systemic issues, rather than general frustrations.
  - Include a requirement for clarity and coherence in the clues, possibly by limiting the length or complexity of individual clues.

- **Reasoning Prompt:**
  - Suggest focusing on 

In [17]:
optimized_clue_prompt = "Please extract specific, detailed, and contextually relevant clues from the interviews that directly relate to each identified topic, particularly focusing on the healthcare system's status before and after the COVID-19 pandemic. Ensure each clue incorporates precise examples, such as specific types of drugs or demographic information that illustrates patient experiences. Accurately quote each clue, highlighting not only key phrases and examples of systemic issues but also their broader implications on public health outcomes. Structure the clues for enhanced clarity and usability, emphasizing their significance to facilitate understanding while maintaining depth and context. Aim to minimize redundancy across clues to strengthen communication effectiveness."

optimized_reasoning_prompt = "Utilizing the extracted clues, develop a structured reasoning process that explicitly links each clue to the identified topics. Clearly delineate how the emotional and psychological states discussed in the clues influence actionable outcomes in healthcare service delivery. Address the implications of these connections within the context of broader health system responses and their impacts on patient care. Support your analysis with specific examples or case studies that illustrate these points, ensuring a thorough and logical connection between the clues and overarching topics. Strive for clarity and conciseness in your reasoning, actively avoiding redundancy to enhance readability and impact, while also deepening the exploration of how these factors affect overall health outcomes."

### Inference -- identify topics for new interviews

To save computational cost, here we only test two new interviews.

In [18]:
def topic_identification_prompt(dialogue, optimized_clue, optimized_reasoning):
    prompt = (
        "Your task is to identify ALL applicable topics for the given interview."
        "Each topic should be concise, meaningful, and specific. Avoid combining distinct ideas or using vague terms.\n\n"
        "There may be multiple topics, so ensure you capture each distinct one.\n\n"
        f"Step 1 Extract CLUES: {optimized_clue}\n\n"
        f"Step 2 Generate REASONING: {optimized_reasoning}\n\n"
        "Step 3 Identify TOPICS: Based on the interview, clues, and reasoning, identify all applicable topics.\n\n"
        "### IMPORTANT REQUIREMENTS FOR IDENTIFIED TOPICS ###\n\n"
        "- **Clarity:** Use precise and specific language, avoiding vague or ambiguous terms\n\n"
        "- **Single Concept:** Ensure each topic represents one distinct idea, avoiding the merging of separate concepts.\n\n"
        "- **Relevance and Specificity:** Make topics meaningful, actionable, and directly related to the context of the interview.\n\n"
        "- **Self-Explanatory:** Each topic should be understandable on its own, without needing to read the clues or reasoning. The topic itself should help readers grasp the content meaningfully.\n\n"
        "### Output Format ###\n\n"
        "For EACH identified topic, provide the following EXACTLY:\n\n"
        "Identify topic: [Insert topic here]\n\n"
        "Clues (max 200 words): [Insert clues here]\n\n"
        "Reasoning (max 150 words): [Insert reasoning here]\n\n"
        f"Dialogue: {dialogue}\n\n"
    )

    messages = [
        {
            "role": "system",
            "content": """You are a qualitative research expert with extensive experience analyzing interviews.
            These interviews were conducted with health workers, policymakers, key informants, and patients between Oct-Dec 2020 to examining the health system response to COVID-19 in Sierra Leone.
            The research aims to explore how the pandemic affected service delivery, health workers, patient access to services, leadership, and governance. Additionally, the research examines to what extent the legacy of the 2013–2016 Ebola outbreak influenced the COVID-19 response and public perception.

            Your task:
            - Identify **all applicable topics** for the given interview (there may be more than one).
            - For each identified topic, provide extracted clues and generated reasoning to explain the connection.

            Clues must:
            - Be direct quotes from the intervew (no summarization or interpretation).
            - Be brief but contextually complete.
            - Highlight key phrases, contextual information, semantic and emotional tones, or temporal information related to the topic.

            Reasoning must:
            - Links the clues directly to the topic.
            - Explains the logical connection between the clues and topic.
            - Avoids adding external context or information not present in the clues.
            """
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
    return messages

In [19]:
def process_multiple_dialogues(dialogue_file_paths, optimized_clue, optimized_reasoning, output_dir, n):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)  # Create output directory if it doesn't exist

    summary = {}

    for dialogue_file_path in dialogue_file_paths:
        dialogue_name = os.path.splitext(os.path.basename(dialogue_file_path))[0]
        output_file_path = os.path.join(output_dir, f"{dialogue_name}_results.json")

        results = {}

        try:
            dialogue = extract_text_from_docx(dialogue_file_path)

            for run in range(1, n+1):
                messages = topic_identification_prompt(dialogue, optimized_clue, optimized_reasoning)
                response = get_completion(messages)
                res = response.choices[0].message.content.strip()
                results[f"run_{run}"] = res

                time.sleep(5)

        except Exception as e:
            print(f"Error processing {dialogue_file_path}: {e}")
            results["error"] = str(e)

        with open(output_file_path, "w") as json_file:
            json.dump(results, json_file, indent=4)

        summary[dialogue_file_path] = output_file_path

    return summary

In [20]:
test_docx_files = [
   "/content/interview/Transcript_B.docx",
   "/content/interview/Transcript_C.docx"
]


output_dir = "/content/testing_results"

# each interview will be processed n times
summary = process_multiple_dialogues(test_docx_files, optimized_clue_prompt, optimized_reasoning_prompt, output_dir, n=3)

In [21]:
def common_topics(json_file_path):
    try:
        with open(json_file_path, "r") as file:
            outputs = json.load(file)
    except Exception as e:
        print(f"Error reading file {json_file_path}: {e}")
        return None

    prompt = (
        "You are analyzing topic identification outputs from multiple analyses of the same interview.\n\n"
        "### Your Goal ###\n\n"
        "Identify common topics across multiple outputs that appear in at least two outputs. For a topic to be considered common, it must:\n"
        "- Have similar meaning.\n"
        "- Be supported by similar extracted clues.\n"
        "- Have similar reasoning.\n\n"
        "**Important:**\n"
        "- Do NOT identify common topics within a single output.\n"
        "- Only compare across multiple outputs.\n\n"

        "### Instructions ###\n"
        "For each common topic:\n"
        "1. Select the best topic name from the outputs that represents the common topic.\n"
        "2. Aggregate all associated clues (without modification).\n"
        "3. Summarize the reasoning concisely.\n\n"

        "### Output Format ###\n"
        "Provide your results in the following format for EACH common topic:\n\n"
        "Topic: [Insert best topic name]\n\n"
        "Clues (max 200 words): [Insert aggregated clues]\n\n"
        "Reasoning (max 150 words): [Insert summarized reasoning]\n\n"
        "If no common topics are found, respond with:\n"
        "'No common topics found.'\n\n"
    )

    for idx, content in enumerate(outputs.values(), start=1):
        prompt += f"Output {idx}:\n{content}\n\n"

    messages = [
        {
            "role": "system",
            "content": "You are an advanced AI designed to analyze outputs from topic identification tasks. Your job is to identify and process common topics based on meaning, clues, and reasoning across multiple outputs."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
    return messages

In [22]:
def process_common_topics(summary, processed_results_dir, common_results_dir):
    os.makedirs(common_results_dir, exist_ok=True)

    common_results_summary = {}

    for dialogue_file_path, json_file_path in summary.items():
        try:
            dialogue_name = os.path.splitext(os.path.basename(dialogue_file_path))[0]
            common_file_path = os.path.join(common_results_dir, f"{dialogue_name}_common.json")

            messages = common_topics(json_file_path)
            response = get_completion(messages)
            commons = response.choices[0].message.content.strip()

            with open(common_file_path, "w") as file:
                json.dump({"common_topics": commons}, file, indent=4)

            common_results_summary[dialogue_file_path] = common_file_path

        except Exception as e:
            print(f"Error consolidating topics for {dialogue_file_path}: {e}")

    return common_results_summary


In [23]:
processed_results_dir = "/content/testing_results"
common_results_dir = "/content/consistent_results"

common_results = process_common_topics(summary, processed_results_dir, common_results_dir)

In [24]:
# clean json files
def remove_double_asterisks(data):
    """
    Recursively remove '**' from all keys and values in a dictionary or list.
    """
    if isinstance(data, dict):
        return {key.replace('**', ''): remove_double_asterisks(value) for key, value in data.items()}
    elif isinstance(data, list):
        return [remove_double_asterisks(item) for item in data]
    elif isinstance(data, str):
        return data.replace('**', '')
    else:
        return data

def remove_triple_hash(data):
    """
    Recursively remove '###' from all keys and values in a dictionary or list.
    """
    if isinstance(data, dict):
        return {key.replace('### ', ''): remove_triple_hash(value) for key, value in data.items()}
    elif isinstance(data, list):
        return [remove_triple_hash(item) for item in data]
    elif isinstance(data, str):
        return data.replace('### ', '')
    else:
        return data

def process_multiple_json_files(file_paths):
    for file_path in file_paths:
        try:
            with open(file_path, 'r') as file:
                json_data = json.load(file)

            # Clean the JSON data
            cleaned_data = remove_double_asterisks(json_data)
            cleaned_data = remove_triple_hash(cleaned_data)

            with open(file_path, 'w') as file:
                json.dump(cleaned_data, file, indent=4)

        except Exception as e:
            print(f"Error processing file {file_path}: {e}")

In [25]:
file_paths = [
    "/content/consistent_results/Transcript_B_common.json",
    "/content/consistent_results/Transcript_C_common.json"
]

process_multiple_json_files(file_paths)

In [26]:
def merge_json_files(file_paths, output_file_path):
    merged_data = {}

    for file_path in file_paths:
        try:
            with open(file_path, 'r') as file:
                json_data = json.load(file)

            file_name = os.path.splitext(os.path.basename(file_path))[0]
            merged_data[file_name] = json_data

        except Exception as e:
            print(f"Error processing file {file_path}: {e}")

    output_dir = os.path.dirname(output_file_path)
    os.makedirs(output_dir, exist_ok=True)

    try:
        with open(output_file_path, 'w') as file:
            json.dump(merged_data, file, indent=4)
    except Exception as e:
        print(f"Error saving merged JSON file: {e}")


file_paths = [
    "/content/consistent_results/Transcript_B_common.json",
    "/content/consistent_results/Transcript_C_common.json"
]

output_file_path = "/content/consistent_results/all_patients_common.json"

merge_json_files(file_paths, output_file_path)

In [27]:
# extract (topic, clues, reasoning) into a table
def parse_common_topics_block(block_text):
    # Extract Topic (everything after "Topic:" until Clues, Reasoning, or end)
    topic_match = re.search(
        r'Topic:\s*(.*?)(?=\nClues|\nReasoning|\Z)',
        block_text,
        flags=re.DOTALL | re.IGNORECASE
    )
    if not topic_match:
        # If there's no "Topic:" line, skip this block entirely
        return None

    topic = topic_match.group(1).strip()

    # Extract Clues (handles optional "(max ... words)")
    clues_match = re.search(
        r'Clues(?:\s*\(max\s*\d+\s*words\))?:\s*(.*?)(?=\nReasoning|\Z)',
        block_text,
        flags=re.DOTALL | re.IGNORECASE
    )
    clues = clues_match.group(1).strip() if clues_match else None

    # Extract Reasoning (handles optional "(max ... words)")
    reasoning_match = re.search(
        r'Reasoning(?:\s*\(max\s*\d+\s*words\))?:\s*(.*)',
        block_text,
        flags=re.DOTALL | re.IGNORECASE
    )
    reasoning = reasoning_match.group(1).strip() if reasoning_match else None

    return {
        "topic": topic,
        "clues": clues,
        "reasoning": reasoning
    }

def parse_multiple_topics_from_text(text, patient_id):
    # Split on lines with '---' (including optional surrounding newlines/spaces)
    blocks = re.split(r'\n\s*---\s*\n', text.strip())

    rows = []
    for block in blocks:
        if not block.strip():
            continue  # skip empty blocks

        parsed = parse_common_topics_block(block)
        if parsed is None:
            # If there's no 'Topic:', skip this block
            continue

        # Add the patient ID and store the row
        parsed["patient_id"] = patient_id
        rows.append(parsed)

    return rows

def parse_common_topics_from_json(json_path):
    with open(json_path, 'r', encoding='utf-8') as infile:
        data = json.load(infile)

    all_rows = []

    for patient_key, sub_dict in data.items():
        if "common_topics" not in sub_dict:
            continue  # skip if missing

        text = sub_dict["common_topics"]

        # Parse multiple topic blocks in this text
        rows = parse_multiple_topics_from_text(text, patient_id=patient_key)
        all_rows.extend(rows)

    # Convert the list of rows to a DataFrame
    df = pd.DataFrame(all_rows, columns=["patient_id", "topic", "clues", "reasoning"])
    return df


In [28]:
df = parse_common_topics_from_json("/content/consistent_results/all_patients_common.json")

In [29]:
df

,patient_id,topic,clues,reasoning
0,Transcript_B_common,Impact of COVID-19 on Patient Access to Services,"- ""the patients in-flow dropped drastically.""\...",The clues illustrate how the COVID-19 pandemic...
1,Transcript_B_common,Changes in Health Worker Responsibilities,"- ""my responsibilities doubled up.""\n- ""I’ve t...",The clues highlight the significant changes in...
2,Transcript_B_common,Health System Preparedness and Response,"- ""the hospital management also called several...",The clues demonstrate the health system's prep...
3,Transcript_B_common,Influence of the Ebola Outbreak on COVID-19 Re...,"- ""I can say that it’s better than the Ebola o...",The clues draw a direct connection between the...
4,Transcript_B_common,Changes in Health Facility Operations,"- ""only emergency patients will be operated on...",The clues illustrate significant changes in he...
5,Transcript_C_common,Impact on Healthcare Service Delivery,"""the only changes that I would say happened in...",The extracted clues from different outputs con...
6,Transcript_C_common,Health Workers’ Experience and Teamwork,"""I saw they all work as a team,"" ""there was te...",The clues across multiple outputs consistently...
7,Transcript_C_common,Public Perception of COVID-19 and Healthcare F...,"""the fear of COVID 19 many patients are not co...",The clues consistently indicate a significant ...
8,Transcript_C_common,Influence of Ebola Experience on COVID-19 Resp...,"""the experience that we had during the EBOLA t...",The clues illustrate how the experience from t...


### Cluster -- develop codebook

In [34]:
def topic_cluster_explain_prompt(dataframe):
    topic_clue_reasoning_blocks = ""

    # Count the total number of original topics
    total_topics = len(dataframe)

    # Construct the topic blocks from the DataFrame
    for i, row in dataframe.iterrows():
        topic_clue_reasoning_blocks += (
            f"Topic {i + 1}: {row['topic']}\n"
            f"Clues: {row['clues']}\n"
            f"Reasoning: {row['reasoning']}\n\n"
        )

    prompt = (
        f"You are given {total_topics} original topics, along with their associated clues and reasoning:\n\n"
        "Your task is to analyze all the topics with the corresponding clues and reasoning. "
        "Identify topics with clues and reasonings that are similar, semantically related, or duplicates, and merge them into several distinct and non-overlapping consolidated topics.\n"
        "If some topics cannot be merged with others, leave them as standalone consolidated topics in the final output.\n"
        "Double-check that all original topics are accounted for in your final output.\n"
        "============\n"
        "### STRICT REQUIREMENTS FOR CONSOLIDATED TOPICS ###\n\n"
        "- **Single Concept:** Each consolidated topic must represent one distinct concept. Do not combine unrelated or separate ideas into a single topic.\n\n"
        "- **Clarity and Specificity:** Consolidated topics should be concise, meaningful, actionable, and specific.\n\n"
        "- **Semantic Similarity:** Only merge topics that are semantically related or synonymous, sharing the same underlying concept.\n\n"
        "- **Preserve Original Meaning:** Ensure the original meaning of each merged topic is maintained without introducing new interpretations.\n\n"
        "- **Standalone Topics:** Topics that cannot be semantically merged with others should remain as standalone consolidated topics.\n\n"
        "- **Complete Representation:** Confirm that all original topics ({total_topics}) are fully represented in the final consolidated output.\n\n"
        "### Output Format ###\n\n"
        "For EACH consolidated topic, provide the following:\n"
        "Consolidated topic: [Insert consolidated topic here]\n\n"
        "Original topics: [Insert original topic names here]\n\n"
        "Explanation (max 150 words): [Insert explanation here]\n\n"
        "============\n\n"
        f"{topic_clue_reasoning_blocks}\n\n"
        )

    messages = [
        {
            "role": "system",
            "content": """
            Your task is to analyze the provided list of topics, along with their associated clues and reasoning, to identify and merge topics that are either similar or duplicates.
            Focus strictly on merging topics that are semantically related or synonymous, sharing the same core concept.
            Under no circumstances should distinct topics be merged into one consolidated topic.
            Topics that cannot be merged with others must remain as standalone consolidated topics.
            Double-check your output to ensure that all original topics are fully represented in the final output. Explicitly mention any topics that were not included in the consolidated topics.
            """
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    return messages

In [35]:
messages = topic_cluster_explain_prompt(df)
response = get_completion(messages)
cluster_explain_text = response.choices[0].message.content
print(cluster_explain_text)

Consolidated topic: Impact of COVID-19 on Patient Access to Healthcare Services

Original topics: Impact of COVID-19 on Patient Access to Services, Impact on Healthcare Service Delivery, Public Perception of COVID-19 and Healthcare Facilities

Explanation: The consolidation of these topics reflects a shared focus on how COVID-19 influenced patient access to healthcare services. All three topics discuss the decline in hospital visits due to patient fears of contracting the virus, leading to a significant impact on healthcare service delivery and public perception. The psychological barriers faced by patients and the resultant changes in healthcare utilization are emphasized across the clues, illustrating the intertwined effects of fear and perception on patient access during the pandemic.

---

Consolidated topic: Changes in Healthcare Worker Responsibilities and Experiences

Original topics: Changes in Health Worker Responsibilities, Health Workers’ Experience and Teamwork

Explanation